# Q1 Consistency + Smoothness Diagnostics (Notebook)

本 Notebook 将原 `run_consistency_smoothness.py` 脚本改为可交互运行版本：
- 读取 `data/processed/inferred_votes_long.csv`
- 计算淘汰一致性（按周）
- 计算时间平滑性扣分（按季）+ 违规明细
- 合成最终分数 + 画热力图

输出文件默认保存到：`outputs/` 与 `outputs/figures/`。


In [ ]:
# === Imports ===
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 120)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)


## 1) 路径设置

如果 notebook 不在项目根目录，可能需要手动调整 `BASE_DIR`。


In [ ]:
# === Paths ===
# 方案A（默认）：假设当前工作目录就是项目根目录
BASE_DIR = Path.cwd()

# 方案B（如果上面不对）：手动指定项目根目录，例如：
# BASE_DIR = Path('/absolute/path/to/your/project')

OUTDIR = BASE_DIR / 'outputs'
FIGDIR = OUTDIR / 'figures'
INPUT_LONG = BASE_DIR / 'data' / 'processed' / 'inferred_votes_long.csv'

OUTDIR, FIGDIR, INPUT_LONG


## 2) 超参数（可改）


In [ ]:
# ---------- Hyperparameters ----------
ALPHA = 0.5          # 如果 S_hat 不存在，会用它计算 S = alpha*J_pct + (1-alpha)*V_hat
ACTIVE_EPS = 1e-12   # 判断“仍在参赛”的阈值
DELTA_THR = 0.15     # 时间平滑阈值 δ：相邻两周 V_hat 差超过该值就扣分
LAMBDA = 0.5         # 综合扣分强度 λ
STRICT_SET_MATCH = True  # 淘汰一致性：True=集合完全相等；False=历史淘汰者是否都在预测bottom-k中
MAX_WEEK = 11

print('ALPHA=', ALPHA, 'ACTIVE_EPS=', ACTIVE_EPS, 'DELTA_THR=', DELTA_THR, 'LAMBDA=', LAMBDA, 'STRICT_SET_MATCH=', STRICT_SET_MATCH, 'MAX_WEEK=', MAX_WEEK)


## 3) 工具函数（与原脚本一致）


In [ ]:
def ensure_outdir():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    FIGDIR.mkdir(parents=True, exist_ok=True)


def active_filter(df_week: pd.DataFrame) -> pd.DataFrame:
    """过滤掉该周不再参赛的记录（通常 J_pct、V_hat 都是 0）"""
    m = (df_week["J_pct"].to_numpy(float) > ACTIVE_EPS) | (df_week["V_hat"].to_numpy(float) > ACTIVE_EPS)
    return df_week.loc[m].copy()


def pick_bottom_k(df_week: pd.DataFrame, score_col: str, k: int) -> list:
    """按得分从小到大选 bottom-k（用名字做 tie-break）"""
    k = int(max(1, k))
    tmp = df_week.sort_values([score_col, "celebrity_name"], ascending=[True, True])
    return tmp["celebrity_name"].head(k).tolist()


def compute_elim_consistency(df: pd.DataFrame) -> pd.DataFrame:
    """
    返回每个淘汰周的：
      - 历史淘汰集合
      - 预测淘汰集合（基于 S_hat 或临时 S）
      - 是否一致 L_{s,t}
    """
    rows = []

    for (s, t), sub_all in df.groupby(["season", "week"]):
        sub_all = sub_all.copy()
        sub = active_filter(sub_all)

        if sub.empty or len(sub) < 2:
            continue

        # 历史淘汰集合
        actual = sub.loc[sub["is_eliminated_this_week"] == 1, "celebrity_name"].tolist()
        k = len(actual)
        if k <= 0:
            continue  # 无淘汰周不进入淘汰一致性统计

        # 预测得分
        if "S_hat" not in sub.columns:
            sub["S_hat"] = ALPHA * sub["J_pct"].to_numpy(float) + (1 - ALPHA) * sub["V_hat"].to_numpy(float)

        pred = pick_bottom_k(sub, "S_hat", k)

        if STRICT_SET_MATCH:
            ok = int(set(pred) == set(actual))
        else:
            ok = int(set(actual).issubset(set(pred)))

        rows.append(dict(
            season=int(s),
            week=int(t),
            n_active=int(len(sub)),
            n_elim=int(k),
            actual_elim=" | ".join(actual),
            pred_elim=" | ".join(pred),
            consistent=ok,
        ))

    return pd.DataFrame(rows)


def compute_smoothness_penalty(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    计算每季平滑性扣分 P_s：
      对同一选手的相邻两周（且两周都 active）计算 |V_hat(t)-V_hat(t-1)|，
      超过 DELTA_THR 的超出部分累计，最后归一化。
    同时输出一个 violations 表便于查哪些人哪些周跳太大。
    """
    df = df.copy()
    df = df[(df["week"] >= 1) & (df["week"] <= MAX_WEEK)].copy()

    # 标记 active
    df["active"] = ((df["J_pct"].to_numpy(float) > ACTIVE_EPS) | (df["V_hat"].to_numpy(float) > ACTIVE_EPS)).astype(int)

    # 按 season, celebrity 排序做差分
    df = df.sort_values(["season", "celebrity_name", "week"]).reset_index(drop=True)

    violations = []
    pen_rows = []

    for s, g_season in df.groupby("season"):
        total_excess = 0.0
        total_pairs = 0

        for name, g in g_season.groupby("celebrity_name"):
            w = g["week"].to_numpy(int)
            v = g["V_hat"].to_numpy(float)
            a = g["active"].to_numpy(int)

            # 遍历相邻行（周序已排序）
            for idx in range(1, len(g)):
                # 要求两周都 active，且周号相邻（t-1, t）
                if a[idx] == 1 and a[idx - 1] == 1 and (w[idx] == w[idx - 1] + 1):
                    dv = abs(v[idx] - v[idx - 1])
                    excess = max(0.0, dv - DELTA_THR)
                    total_excess += excess
                    total_pairs += 1

                    if excess > 0:
                        violations.append(dict(
                            season=int(s),
                            celebrity_name=name,
                            week_prev=int(w[idx - 1]),
                            week=int(w[idx]),
                            V_prev=float(v[idx - 1]),
                            V_now=float(v[idx]),
                            delta=float(dv),
                            excess=float(excess),
                        ))

        # 归一化到 [0,1]（理论最大超出为 1-DELTA_THR）
        if total_pairs == 0:
            P = np.nan
        else:
            P = (total_excess / total_pairs) / max(1e-12, (1.0 - DELTA_THR))
            P = float(np.clip(P, 0.0, 1.0))

        pen_rows.append(dict(
            season=int(s),
            smooth_pairs=int(total_pairs),
            smooth_penalty=P,                # P_s
            smooth_score=(1 - P) if P == P else np.nan  # S_s^{smooth}
        ))

    pen_df = pd.DataFrame(pen_rows)
    vio_df = pd.DataFrame(violations)
    return pen_df, vio_df


def build_season_scores(elim_week_df: pd.DataFrame, smooth_df: pd.DataFrame) -> pd.DataFrame:
    # L_s
    if elim_week_df.empty:
        raise ValueError("没有检测到淘汰周，请检查 is_eliminated_this_week 是否正确。")

    Ls = (elim_week_df.groupby("season", as_index=False)
          .agg(elim_weeks=("week", "count"),
               elim_consistency=("consistent", "mean")))

    out = Ls.merge(smooth_df, on="season", how="left")

    # 综合 Score_s = clip(L_s - λ P_s, 0, 1)
    P = out["smooth_penalty"].to_numpy(float)
    L = out["elim_consistency"].to_numpy(float)
    score = L - LAMBDA * P
    score = np.clip(score, 0.0, 1.0)

    out["lambda"] = LAMBDA
    out["delta_thr"] = DELTA_THR
    out["final_score"] = score
    return out.sort_values("season")


def plot_season_scores(season_df: pd.DataFrame) -> Path:
    """
    热力图：3 行 × 多赛季列
      行1: elimination consistency L_s
      行2: smoothness penalty P_s (higher = worse)
      行3: final score Score_s = L_s - lambda P_s
    """
    seasons = season_df["season"].astype(int).to_numpy()
    L = season_df["elim_consistency"].to_numpy(float)
    P = season_df["smooth_penalty"].to_numpy(float)
    Score = season_df["final_score"].to_numpy(float)

    # 若 P 有 NaN（没有可比较的相邻周），用 0 填充
    P_filled = np.where(np.isfinite(P), P, 0.0)

    # 组装 3×S 矩阵
    M = np.vstack([L, P_filled, Score])
    row_labels = [r"$L_s$ (elim. consistency)", r"$P_s$ (smooth penalty)", r"$Score_s=L_s-\lambda P_s$"]

    plt.figure(figsize=(14, 3.8))
    ax = plt.gca()
    im = ax.imshow(M, aspect="auto")

    ax.set_yticks(np.arange(3))
    ax.set_yticklabels(row_labels)

    ax.set_xticks(np.arange(len(seasons))[::2])
    ax.set_xticklabels([str(s) for s in seasons[::2]], rotation=0)

    ax.set_xlabel("Season")
    ax.set_title(f"Q1 Consistency & Smoothness diagnostics (delta={DELTA_THR}, lambda={LAMBDA})")

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Value")

    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            val = M[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)

    ax.set_yticks(np.arange(-.5, 3, 1), minor=True)
    ax.grid(which="minor", axis="y", linewidth=1)
    ax.tick_params(which="minor", left=False)

    plt.tight_layout()
    path = FIGDIR / 'q1_consistency_with_smoothness.png'
    plt.savefig(path, dpi=220)
    plt.show()
    plt.close()
    return path


## 4) 读取数据并检查


In [ ]:
ensure_outdir()

if not INPUT_LONG.exists():
    raise FileNotFoundError(f"找不到 {INPUT_LONG}，请先运行第一问脚本生成 inferred_votes_long.csv")

df = pd.read_csv(INPUT_LONG, encoding='utf-8-sig')
print('Loaded:', df.shape)
df.head()


In [ ]:
required = {"season", "week", "celebrity_name", "J_pct", "V_hat", "is_eliminated_this_week"}
miss = required - set(df.columns)
if miss:
    raise ValueError(f"{INPUT_LONG} 缺少列：{miss}")

df["season"] = df["season"].astype(int)
df["week"] = df["week"].astype(int)
df["J_pct"] = pd.to_numeric(df["J_pct"], errors="coerce").fillna(0.0)
df["V_hat"] = pd.to_numeric(df["V_hat"], errors="coerce").fillna(0.0)
df["is_eliminated_this_week"] = pd.to_numeric(df["is_eliminated_this_week"], errors="coerce").fillna(0).astype(int)

df[["season","week","celebrity_name","J_pct","V_hat","is_eliminated_this_week"]].head()


## 5) (1) 淘汰一致性（按周）


In [ ]:
elim_week_df = compute_elim_consistency(df)
elim_week_path = OUTDIR / 'q1_elim_consistency_by_week.csv'
elim_week_df.to_csv(elim_week_path, index=False, encoding='utf-8-sig')

print('Saved:', elim_week_path)
elim_week_df.head(20)


In [ ]:
# 快速查看不一致的周（consistent=0）
bad = elim_week_df[elim_week_df['consistent'] == 0].sort_values(['season','week'])
print('Inconsistent rows:', len(bad))
bad.head(50)


## 6) (2) 平滑性扣分（按季）+ 违规明细


In [ ]:
smooth_df, vio_df = compute_smoothness_penalty(df)
smooth_path = OUTDIR / 'q1_smoothness_by_season.csv'
vio_path = OUTDIR / 'q1_smoothness_violations.csv'
smooth_df.to_csv(smooth_path, index=False, encoding='utf-8-sig')
vio_df.to_csv(vio_path, index=False, encoding='utf-8-sig')

print('Saved:', smooth_path)
print('Saved:', vio_path)
smooth_df.head(20)


In [ ]:
# 看看平滑性违规（excess）最大的一些记录
if len(vio_df) == 0:
    print('No violations (excess>0) found under current DELTA_THR.')
else:
    vio_df_sorted = vio_df.sort_values(['excess','delta'], ascending=[False, False])
    vio_df_sorted.head(30)


## 7) (3) 合成赛季分数 + 保存 + 绘图


In [ ]:
season_df = build_season_scores(elim_week_df, smooth_df)
season_path = OUTDIR / 'q1_consistency_with_smoothness_by_season.csv'
season_df.to_csv(season_path, index=False, encoding='utf-8-sig')

print('Saved:', season_path)
season_df.head(20)


In [ ]:
fig_path = plot_season_scores(season_df)
print('Saved figure:', fig_path)


## 8) 解释与快速结论

- **L_s**：历史淘汰一致性（越高越好）
- **P_s**：时间不平滑惩罚（越高越糟糕）
- **final_score = clip(L_s - λ·P_s, 0, 1)**：在淘汰一致性基础上扣除时间不平滑惩罚后的综合分

你可以在上方超参数 cell 调整 `DELTA_THR` 或 `LAMBDA` 再重跑，观察热力图与表格变化。
